# Modulo 2 · Lezione 5 — Progetto: versionare il sistema di biblioteca

**Corso di Intelligenza Artificiale — Livello base**

---

## Obiettivo del progetto

In questo progetto integratore applichiamo **tutto quello che abbiamo imparato nel Modulo 2** al sistema di biblioteca che abbiamo sviluppato nella Lezione 12 del Modulo 1.

Andremo a:

1. Inizializzare un repository Git per il sistema di biblioteca.
2. Costruire la sua cronologia in modo professionale (commit convenzionali, messaggi chiari).
3. Usare branch per ogni funzionalità.
4. Etichettare le versioni con `git tag`.
5. Pubblicare su GitHub con la struttura corretta.
6. Generare report della cronologia con Python.

```
STRUTTURA DEL PROGETTO:

sistema_biblioteca/
├── excepciones.py      # Eccezioni del dominio
├── material.py         # ABC Material + Libro, Revista, DVD
├── usuario.py          # Classe Usuario
├── prestamo.py         # Classe Prestamo
├── biblioteca.py       # Classe Biblioteca (nucleo)
├── persistencia.py     # Salvataggio/caricamento JSON
├── cli.py              # Menu interattivo
├── tests/
│   └── test_biblioteca.py
├── .gitignore
├── README.md
└── requirements.txt
```

> 💡 Questo progetto simula una cronologia realistica di 4 settimane di sviluppo, dove ogni commit racconta una storia chiara.

---
## Fase 1 — Inizializzare il repository

Iniziamo da zero: una directory vuota che diventa un progetto Git professionale.

In [ ]:
import os, shutil
from pathlib import Path

BLIB = Path.home() / "sistema_biblioteca"

if BLIB.exists():
    shutil.rmtree(BLIB)

BLIB.mkdir()
(BLIB / "tests").mkdir()
os.chdir(BLIB)

# Inizializzare Git
!git init -b main
!git config user.name "Alumno Curso"
!git config user.email "alumno@curso.io"

print(f"Directory: {Path.cwd()}")

In [ ]:
# .gitignore professionale per Python
Path(".gitignore").write_text("""\
# Python
__pycache__/
*.py[cod]
*.pyo
*.pyd
.Python

# Ambienti virtuali
venv/
env/
.venv/

# Dati (non versionare file di dati generati)
datos/
*.json
!package.json

# IDE
.vscode/
.idea/
*.swp

# macOS
.DS_Store

# Test
.pytest_cache/
htmlcov/
.coverage
""", encoding="utf-8")

!git add .gitignore
!git commit -m "chore: aggiungere .gitignore per Python"
print("Primo commit creato")

In [ ]:
Path("README.md").write_text("""\
# Sistema di Gestione Biblioteca

Sistema per gestire il catalogo, gli utenti e i prestiti di una biblioteca.
Progetto integratore del Modulo 1 del Corso di IA.

## Installazione

```bash
git clone https://github.com/TUO-UTENTE/sistema-biblioteca.git
cd sistema-biblioteca
pip install -r requirements.txt
```

## Uso

```bash
python cli.py
```

## Architettura

```
Material (ABC)
├── Libro
├── Revista
└── DVD

Biblioteca
├── catalogo: list[Material]
├── usuarios: list[Usuario]
└── prestamos: list[Prestamo]
```

## Licenza

MIT
""", encoding="utf-8")

Path("requirements.txt").write_text("# Nessuna dipendenza esterna\n", encoding="utf-8")

!git add README.md requirements.txt
!git commit -m "docs: aggiungere README con architettura e requirements"

print("\nCronologia iniziale:")
!git log --oneline

---
## Fase 2 — Sviluppare il nucleo su branch

Ogni modulo del sistema viene sviluppato nel proprio branch, come farebbe un team reale.

In [ ]:
# feature/excepciones — il sistema di eccezioni del dominio
!git switch -c feature/excepciones

Path("excepciones.py").write_text("""\
class BibliotecaError(Exception):
    pass

class MaterialNoEncontrado(BibliotecaError):
    def __init__(self, isbn):
        super().__init__(f"Material con ISBN '{isbn}' no encontrado")
        self.isbn = isbn

class UsuarioNoEncontrado(BibliotecaError):
    def __init__(self, id_usuario):
        super().__init__(f"Usuario con ID '{id_usuario}' no encontrado")
        self.id_usuario = id_usuario

class MaterialNoDisponible(BibliotecaError):
    def __init__(self, isbn):
        super().__init__(f"Material '{isbn}' no disponible para préstamo")
        self.isbn = isbn

class LimitePrestamosAlcanzado(BibliotecaError):
    def __init__(self, id_usuario, limite):
        super().__init__(f"Usuario '{id_usuario}' ha alcanzado el límite de {limite} préstamos")
        self.id_usuario = id_usuario
        self.limite = limite

class PrestamoNoEncontrado(BibliotecaError):
    def __init__(self, id_prestamo):
        super().__init__(f"Préstamo '{id_prestamo}' no encontrado")
        self.id_prestamo = id_prestamo
""", encoding="utf-8")

!git add excepciones.py
!git commit -m "feat: gerarchia di eccezioni del dominio biblioteca"

print("Branch feature/excepciones pronto")

In [ ]:
# Integrare in main
!git switch main
!git merge feature/excepciones --no-ff -m "Merge: gerarchia di eccezioni"
!git branch -d feature/excepciones

print("Cronologia:")
!git log --oneline --graph

In [ ]:
# feature/materiales — gerarchia di materiali con ABC
!git switch -c feature/materiales

Path("material.py").write_text("""\
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Material(ABC):
    isbn: str
    titulo: str
    anio: int
    disponible: bool = field(default=True, init=False)

    @property
    @abstractmethod
    def tipo(self) -> str:
        pass

    @abstractmethod
    def descripcion_corta(self) -> str:
        pass

    def to_dict(self) -> dict:
        return {
            "tipo": self.tipo,
            "isbn": self.isbn,
            "titulo": self.titulo,
            "anio": self.anio,
            "disponible": self.disponible,
        }

    def __str__(self):
        estado = "disponible" if self.disponible else "prestado"
        return f"[{self.tipo}] {self.titulo} ({self.anio}) — {estado}"


@dataclass
class Libro(Material):
    autor: str = ""
    paginas: int = 0

    @property
    def tipo(self) -> str:
        return "Libro"

    def descripcion_corta(self) -> str:
        return f"{self.titulo} — {self.autor} ({self.paginas} págs.)"

    def to_dict(self) -> dict:
        d = super().to_dict()
        d.update({"autor": self.autor, "paginas": self.paginas})
        return d


@dataclass
class Revista(Material):
    numero: int = 0
    editorial: str = ""

    @property
    def tipo(self) -> str:
        return "Revista"

    def descripcion_corta(self) -> str:
        return f"{self.titulo} #{self.numero} — {self.editorial}"

    def to_dict(self) -> dict:
        d = super().to_dict()
        d.update({"numero": self.numero, "editorial": self.editorial})
        return d


@dataclass
class DVD(Material):
    director: str = ""
    duracion_min: int = 0

    @property
    def tipo(self) -> str:
        return "DVD"

    def descripcion_corta(self) -> str:
        return f"{self.titulo} — dir. {self.director} ({self.duracion_min} min)"

    def to_dict(self) -> dict:
        d = super().to_dict()
        d.update({"director": self.director, "duracion_min": self.duracion_min})
        return d


TIPOS_MATERIAL = {
    "Libro": Libro,
    "Revista": Revista,
    "DVD": DVD,
}

def material_from_dict(d: dict) -> Material:
    tipo = d.pop("tipo")
    disponible = d.pop("disponible", True)
    clase = TIPOS_MATERIAL[tipo]
    m = clase(**d)
    m.disponible = disponible
    return m
""", encoding="utf-8")

!git add material.py
!git commit -m "feat: gerarchia Material (ABC) → Libro, Revista, DVD"

# Integrare in main
!git switch main
!git merge feature/materiales --no-ff -m "Merge: gerarchia dei materiali"
!git branch -d feature/materiales

print("feature/materiales integrata")

In [ ]:
# feature/usuarios-y-prestamos
!git switch -c feature/usuarios-y-prestamos

Path("usuario.py").write_text("""\
from dataclasses import dataclass, field

@dataclass
class Usuario:
    id_usuario: str
    nombre: str
    email: str
    prestamos_activos: list = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "id_usuario": self.id_usuario,
            "nombre": self.nombre,
            "email": self.email,
        }

    @classmethod
    def from_dict(cls, d: dict):
        return cls(
            id_usuario=d["id_usuario"],
            nombre=d["nombre"],
            email=d["email"],
        )

    def __str__(self):
        return f"Usuario({self.id_usuario}) {self.nombre} <{self.email}>"
""", encoding="utf-8")

!git add usuario.py
!git commit -m "feat: classe Usuario con serializzazione dict"

Path("prestamo.py").write_text("""\
from dataclasses import dataclass, field
from datetime import date

@dataclass
class Prestamo:
    id_prestamo: str
    id_usuario: str
    isbn_material: str
    fecha_prestamo: date = field(default_factory=date.today)
    fecha_devolucion: date | None = None
    activo: bool = True

    def devolver(self):
        self.fecha_devolucion = date.today()
        self.activo = False

    @property
    def dias_prestado(self) -> int:
        fin = self.fecha_devolucion or date.today()
        return (fin - self.fecha_prestamo).days

    def to_dict(self) -> dict:
        return {
            "id_prestamo": self.id_prestamo,
            "id_usuario": self.id_usuario,
            "isbn_material": self.isbn_material,
            "fecha_prestamo": self.fecha_prestamo.isoformat(),
            "fecha_devolucion": self.fecha_devolucion.isoformat() if self.fecha_devolucion else None,
            "activo": self.activo,
        }

    @classmethod
    def from_dict(cls, d: dict):
        p = cls(
            id_prestamo=d["id_prestamo"],
            id_usuario=d["id_usuario"],
            isbn_material=d["isbn_material"],
            fecha_prestamo=date.fromisoformat(d["fecha_prestamo"]),
            activo=d["activo"],
        )
        if d["fecha_devolucion"]:
            p.fecha_devolucion = date.fromisoformat(d["fecha_devolucion"])
        return p
""", encoding="utf-8")

!git add prestamo.py
!git commit -m "feat: classe Prestamo con gestione delle date"

!git switch main
!git merge feature/usuarios-y-prestamos --no-ff -m "Merge: moduli Usuario e Prestamo"
!git branch -d feature/usuarios-y-prestamos
print("feature/usuarios-y-prestamos integrata")

In [ ]:
# feature/biblioteca — il nucleo dell'applicazione
!git switch -c feature/biblioteca

Path("biblioteca.py").write_text("""\
import uuid
from material import Material, TIPOS_MATERIAL
from usuario import Usuario
from prestamo import Prestamo
from excepciones import (
    MaterialNoEncontrado, UsuarioNoEncontrado,
    MaterialNoDisponible, LimitePrestamosAlcanzado,
    PrestamoNoEncontrado,
)

LIMITE_PRESTAMOS = 3

class Biblioteca:
    def __init__(self, nombre: str):
        self.nombre = nombre
        self._catalogo: list[Material] = []
        self._usuarios: list[Usuario] = []
        self._prestamos: list[Prestamo] = []

    # ── Catalogo ──────────────────────────────────────────────
    def agregar_material(self, material: Material) -> None:
        self._catalogo.append(material)

    def buscar_material(self, isbn: str) -> Material:
        for m in self._catalogo:
            if m.isbn == isbn:
                return m
        raise MaterialNoEncontrado(isbn)

    def buscar_por_titulo(self, termino: str) -> list[Material]:
        t = termino.lower()
        return [m for m in self._catalogo if t in m.titulo.lower()]

    def listar_catalogo(self) -> list[Material]:
        return list(self._catalogo)

    # ── Usuarios ──────────────────────────────────────────────
    def registrar_usuario(self, usuario: Usuario) -> None:
        self._usuarios.append(usuario)

    def buscar_usuario(self, id_usuario: str) -> Usuario:
        for u in self._usuarios:
            if u.id_usuario == id_usuario:
                return u
        raise UsuarioNoEncontrado(id_usuario)

    # ── Préstamos ─────────────────────────────────────────────
    def prestar(self, id_usuario: str, isbn: str) -> Prestamo:
        usuario = self.buscar_usuario(id_usuario)
        material = self.buscar_material(isbn)

        if not material.disponible:
            raise MaterialNoDisponible(isbn)

        prestamos_activos = sum(1 for p in self._prestamos
                                if p.id_usuario == id_usuario and p.activo)
        if prestamos_activos >= LIMITE_PRESTAMOS:
            raise LimitePrestamosAlcanzado(id_usuario, LIMITE_PRESTAMOS)

        material.disponible = False
        prestamo = Prestamo(
            id_prestamo=str(uuid.uuid4())[:8],
            id_usuario=id_usuario,
            isbn_material=isbn,
        )
        self._prestamos.append(prestamo)
        return prestamo

    def devolver(self, id_prestamo: str) -> Prestamo:
        for p in self._prestamos:
            if p.id_prestamo == id_prestamo and p.activo:
                p.devolver()
                material = self.buscar_material(p.isbn_material)
                material.disponible = True
                return p
        raise PrestamoNoEncontrado(id_prestamo)

    def prestamos_activos(self) -> list[Prestamo]:
        return [p for p in self._prestamos if p.activo]

    def reporte_estado(self) -> dict:
        disponibles = sum(1 for m in self._catalogo if m.disponible)
        prestados = len(self._catalogo) - disponibles
        return {
            "total_materiales": len(self._catalogo),
            "disponibles": disponibles,
            "prestados": prestados,
            "total_usuarios": len(self._usuarios),
            "prestamos_activos": len(self.prestamos_activos()),
            "prestamos_totales": len(self._prestamos),
        }
""", encoding="utf-8")

!git add biblioteca.py
!git commit -m "feat: classe Biblioteca con catalogo, utenti e prestiti"

!git switch main
!git merge feature/biblioteca --no-ff -m "Merge: nucleo Biblioteca"
!git branch -d feature/biblioteca

print("\nCronologia dopo la Fase 2:")
!git log --oneline --graph

---
## Fase 3 — Persistenza, CLI e prima versione stabile

In [ ]:
!git switch -c feature/persistencia

Path("persistencia.py").write_text("""\
import json
from pathlib import Path
from biblioteca import Biblioteca
from material import material_from_dict
from usuario import Usuario
from prestamo import Prestamo

def guardar(biblioteca: Biblioteca, ruta: str = "datos.json") -> None:
    datos = {
        "nombre": biblioteca.nombre,
        "catalogo": [m.to_dict() for m in biblioteca.listar_catalogo()],
        "usuarios": [u.to_dict() for u in biblioteca._usuarios],
        "prestamos": [p.to_dict() for p in biblioteca._prestamos],
    }
    Path(ruta).write_text(json.dumps(datos, ensure_ascii=False, indent=2),
                           encoding="utf-8")

def cargar(ruta: str = "datos.json") -> Biblioteca:
    texto = Path(ruta).read_text(encoding="utf-8")
    datos = json.loads(texto)
    bib = Biblioteca(datos["nombre"])
    for d in datos["catalogo"]:
        bib.agregar_material(material_from_dict(d.copy()))
    for d in datos["usuarios"]:
        bib.registrar_usuario(Usuario.from_dict(d))
    for d in datos["prestamos"]:
        bib._prestamos.append(Prestamo.from_dict(d))
    return bib
""", encoding="utf-8")

!git add persistencia.py
!git commit -m "feat: persistenza JSON — salvare e caricare la biblioteca"

!git switch main
!git merge feature/persistencia --no-ff -m "Merge: persistenza JSON"
!git branch -d feature/persistencia
print("feature/persistencia integrata")

In [ ]:
!git switch -c feature/cli

Path("cli.py").write_text("""\
from biblioteca import Biblioteca
from material import Libro, Revista, DVD
from usuario import Usuario
from excepciones import BibliotecaError
import persistencia

ARCHIVO = "datos.json"

MENU = """
╔══════════════════════════════╗
║  SISTEMA DE BIBLIOTECA       ║
╠══════════════════════════════╣
║  1. Ver catálogo             ║
║  2. Añadir libro             ║
║  3. Registrar usuario        ║
║  4. Prestar material         ║
║  5. Devolver material        ║
║  6. Ver préstamos activos    ║
║  7. Ver reporte              ║
║  0. Salir (guardar)          ║
╚══════════════════════════════╝
"""

def main():
    from pathlib import Path
    if Path(ARCHIVO).exists():
        bib = persistencia.cargar(ARCHIVO)
        print(f"Biblioteca '{bib.nombre}' cargada.")
    else:
        nombre = input("Nombre de la biblioteca: ").strip() or "Mi Biblioteca"
        bib = Biblioteca(nombre)

    while True:
        print(MENU)
        opcion = input("Opción: ").strip()
        try:
            if opcion == "0":
                persistencia.guardar(bib, ARCHIVO)
                print("Guardado. ¡Hasta pronto!")
                break
            elif opcion == "1":
                for m in bib.listar_catalogo():
                    print(" ", m)
            elif opcion == "2":
                isbn  = input("ISBN: ").strip()
                titulo = input("Título: ").strip()
                autor  = input("Autor: ").strip()
                anio   = int(input("Año: ").strip())
                bib.agregar_material(Libro(isbn, titulo, anio, autor))
                print("Libro añadido.")
            elif opcion == "3":
                id_u  = input("ID usuario: ").strip()
                nombre = input("Nombre: ").strip()
                email  = input("Email: ").strip()
                bib.registrar_usuario(Usuario(id_u, nombre, email))
                print("Usuario registrado.")
            elif opcion == "4":
                id_u = input("ID usuario: ").strip()
                isbn = input("ISBN: ").strip()
                p = bib.prestar(id_u, isbn)
                print(f"Préstamo {p.id_prestamo} creado.")
            elif opcion == "5":
                id_p = input("ID préstamo: ").strip()
                bib.devolver(id_p)
                print("Devolución registrada.")
            elif opcion == "6":
                for p in bib.prestamos_activos():
                    print(f"  {p.id_prestamo}: usuario={p.id_usuario} isbn={p.isbn_material} desde={p.fecha_prestamo}")
            elif opcion == "7":
                r = bib.reporte_estado()
                for k, v in r.items():
                    print(f"  {k}: {v}")
        except BibliotecaError as e:
            print(f"  ⚠️  {e}")
        except ValueError as e:
            print(f"  ⚠️  Valor incorrecto: {e}")

if __name__ == "__main__":
    main()
""", encoding="utf-8")

!git add cli.py
!git commit -m "feat: CLI interattiva con menu completo"

!git switch main
!git merge feature/cli --no-ff -m "Merge: interfaccia a riga di comando"
!git branch -d feature/cli

# Etichettare la prima versione stabile
!git tag -a v1.0.0 -m "v1.0.0 — Prima versione stabile del sistema"

print("\n✅ v1.0.0 etichettata")
print("\nCronologia completa:")
!git log --oneline --graph

---
## Fase 4 — Aggiungere test ed etichettare v1.1.0

In [ ]:
!git switch -c feature/tests

Path("tests/test_biblioteca.py").write_text("""\
import sys
sys.path.insert(0, str(__import__('pathlib').Path(__file__).parent.parent))

from biblioteca import Biblioteca
from material import Libro
from usuario import Usuario
from excepciones import MaterialNoDisponible, LimitePrestamosAlcanzado

def crear_bib_demo():
    bib = Biblioteca("Test")
    bib.agregar_material(Libro("ISBN-001", "Python Básico", 2023, "Autor A"))
    bib.agregar_material(Libro("ISBN-002", "Git Avanzado", 2022, "Autor B"))
    bib.registrar_usuario(Usuario("U1", "Ana", "ana@test.com"))
    return bib

def test_prestar_ok():
    bib = crear_bib_demo()
    p = bib.prestar("U1", "ISBN-001")
    assert p.activo
    assert not bib.buscar_material("ISBN-001").disponible
    print("✅ test_prestar_ok")

def test_prestar_no_disponible():
    bib = crear_bib_demo()
    bib.prestar("U1", "ISBN-001")
    try:
        bib.prestar("U1", "ISBN-001")
        assert False, "Debería lanzar MaterialNoDisponible"
    except MaterialNoDisponible:
        print("✅ test_prestar_no_disponible")

def test_devolver_ok():
    bib = crear_bib_demo()
    p = bib.prestar("U1", "ISBN-001")
    bib.devolver(p.id_prestamo)
    assert bib.buscar_material("ISBN-001").disponible
    print("✅ test_devolver_ok")

def test_limite_prestamos():
    bib = Biblioteca("Test")
    for i in range(4):
        bib.agregar_material(Libro(f"ISBN-{i:03d}", f"Libro {i}", 2023, "Autor"))
    bib.registrar_usuario(Usuario("U1", "Ana", "ana@test.com"))
    for i in range(3):  # prendere fino al limite
        bib.prestar("U1", f"ISBN-{i:03d}")
    try:
        bib.prestar("U1", "ISBN-003")  # superare il limite
        assert False, "Debería lanzar LimitePrestamosAlcanzado"
    except LimitePrestamosAlcanzado:
        print("✅ test_limite_prestamos")

if __name__ == "__main__":
    test_prestar_ok()
    test_prestar_no_disponible()
    test_devolver_ok()
    test_limite_prestamos()
    print("\n✅ Todos los tests pasaron")
""", encoding="utf-8")

!git add tests/test_biblioteca.py
!git commit -m "test: suite base di test per Biblioteca"

# Eseguire i test per verificare
print("\nEsecuzione dei test:")
!python tests/test_biblioteca.py

!git switch main
!git merge feature/tests --no-ff -m "Merge: suite di test"
!git branch -d feature/tests

!git tag -a v1.1.0 -m "v1.1.0 — Aggiungere suite di test"
print("\n✅ v1.1.0 etichettata")

---
## Pratica — Audit ed estensione del progetto

Completate le fasi guidate, ora lavora in autonomia sugli esercizi.

---

### Esercizio 1 — Audit della cronologia

**Consegna:** Scrivi una funzione `auditar_historial(ruta)` che analizzi la cronologia del progetto e generi un report con:

1. Numero totale di commit.
2. Distribuzione per tipo (feat, fix, docs, test, chore, Merge) con conteggio e percentuale.
3. Commit che NON seguono il formato Conventional Commits (senza prefisso `tipo:`).
4. Elenco dei tag con la data di ogni versione.
5. Numero di file unici modificati in tutta la cronologia.

In [ ]:
# ✏️ IL TUO CODICE QUI
import subprocess, os
from pathlib import Path

os.chdir(Path.home() / "sistema_biblioteca")
print(f"Lavorando in: {Path.cwd()}")


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import subprocess, re
from pathlib import Path
from collections import Counter

def auditar_historial(ruta):
    ruta = Path(ruta).expanduser().resolve()

    def git(args):
        r = subprocess.run(["git"] + args, cwd=str(ruta),
                           capture_output=True, text=True)
        return r.stdout.strip()

    # 1. Commit totali
    logs = git(["log", "--pretty=format:%H|%s"]).splitlines()
    total = len(logs)

    # 2. Distribuzione per tipo
    prefijos_validos = ("feat", "fix", "docs", "test", "chore", "refactor", "style")
    tipos = Counter()
    sin_prefijo = []
    for linea in logs:
        _, msg = linea.split("|", 1)
        if msg.startswith("Merge"):
            tipos["Merge"] += 1
            continue
        match = re.match(r'^(\w+)[:(]', msg)
        if match and match.group(1) in prefijos_validos:
            tipos[match.group(1)] += 1
        else:
            tipos["otros"] += 1
            sin_prefijo.append(msg)

    # 4. Tag
    tags_raw = git(["tag", "-l", "--sort=version:refname"]).splitlines()
    tags_info = []
    for tag in tags_raw:
        fecha = git(["log", "-1", "--format=%ad", "--date=short", tag])
        tags_info.append((tag, fecha))

    # 5. File unici modificati
    archivos_raw = git(["log", "--name-only", "--pretty=format:"]).splitlines()
    archivos = {a for a in archivos_raw if a.strip()}

    # Stampare il report
    print(f"\n{'═'*55}")
    print(f"  AUDIT DELLA CRONOLOGIA — {ruta.name}")
    print(f"{'═'*55}")
    print(f"  Totale commit: {total}")

    print(f"\n  DISTRIBUZIONE PER TIPO:")
    for tipo, count in tipos.most_common():
        pct = count / total * 100
        barra = "█" * int(pct / 5)
        print(f"    {tipo:<12} {count:>3}  {pct:5.1f}%  {barra}")

    print(f"\n  COMMIT SENZA PREFISSO CONVENZIONALE ({len(sin_prefijo)}):")
    for m in sin_prefijo:
        print(f"    - {m[:60]}")
    if not sin_prefijo:
        print("    ✅ Tutti i commit seguono Conventional Commits")

    print(f"\n  VERSIONI (tag):")
    for tag, fecha in tags_info:
        print(f"    {tag:<12}  {fecha}")

    print(f"\n  File unici modificati: {len(archivos)}")
    for a in sorted(archivos):
        print(f"    {a}")

    print(f"{'═'*55}")


auditar_historial("~/sistema_biblioteca")
```
</details>

---
### Esercizio 2 — Hotfix su v1.0.0

**Consegna:** Scopri che in `v1.0.0` c'è un bug: `buscar_por_titulo` è sensibile alle maiuscole. Applica il flusso di hotfix:

1. Crea il branch `hotfix/busqueda-case-insensitive` da `v1.0.0` (non da `main`).
2. Correggi il bug in `biblioteca.py`: la ricerca non distingue più tra maiuscole/minuscole.
3. Committa con un messaggio appropriato.
4. Fondi l'hotfix in `main`.
5. Etichetta `v1.0.1`.
6. Mostra la cronologia grafica.

In [ ]:
# ✏️ IL TUO CODICE QUI
# Suggerimento: git switch -c hotfix/... v1.0.0


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import os, subprocess
from pathlib import Path

os.chdir(Path.home() / "sistema_biblioteca")

# 1. Creare l'hotfix dal tag v1.0.0
!git switch -c hotfix/busqueda-case-insensitive v1.0.0

# 2. Correggere il bug
# buscar_por_titulo usava già .lower() → in realtà va bene.
# Aggiungiamo anche la ricerca nel tipo e nella descripcion_corta.
contenido = Path("biblioteca.py").read_text(encoding="utf-8")
contenido = contenido.replace(
    'return [m for m in self._catalogo if t in m.titulo.lower()]',
    'return [m for m in self._catalogo if t in m.titulo.lower() or t in m.tipo.lower()]'
)
Path("biblioteca.py").write_text(contenido, encoding="utf-8")

# 3. Commit
!git add biblioteca.py
!git commit -m "fix: buscar_por_titulo include ricerca nel tipo di materiale"

# 4. Fondere in main
!git switch main
!git merge hotfix/busqueda-case-insensitive --no-ff -m "Merge: hotfix ricerca case-insensitive"
!git branch -d hotfix/busqueda-case-insensitive

# 5. Etichettare
!git tag -a v1.0.1 -m "v1.0.1 — Hotfix: ricerca per tipo di materiale"

# 6. Cronologia
!git log --oneline --graph
print("\nTag:")
!git tag -l
```

**Nota:** In un Git flow reale, l'hotfix verrebbe fuso anche in `develop` (se esistesse). Nei progetti piccoli con solo `main`, basta fondere in `main`.
</details>

---
### Esercizio 3 — Nuova feature: statistiche

**Consegna:** Aggiungi al sistema una funzionalità di statistiche seguendo il flusso di feature branch:

1. Crea il branch `feature/estadisticas` da `main`.
2. Aggiungi il metodo `estadisticas_avanzadas()` in `Biblioteca` che restituisca:
   - Materiale più prestato (isbn, titolo, volte_prestato)
   - Utente con più prestiti storici
   - Media di giorni per prestito restituito
3. Committa.
4. Fondi in `main` con `--no-ff`.
5. Etichetta `v1.2.0`.

In [ ]:
# ✏️ IL TUO CODICE QUI


---
### Esercizio 4 — Pubblicare su GitHub con struttura completa

**Consegna:** Pubblica il progetto `sistema_biblioteca` su GitHub:

1. Crea il repository su GitHub (senza inizializzarlo).
2. Aggiungi il remote `origin`.
3. Carica `main` con tutti i suoi commit.
4. Carica tutti i tag con `git push --tags`.
5. Verifica su GitHub che:
   - Appaiano tutti i branch (solo `main` — le feature branch sono state eliminate)
   - Appaiano i tag `v1.0.0`, `v1.0.1`, `v1.1.0`, `v1.2.0`
   - La cronologia mostri il grafo dei merge
6. Scrivi l'URL del repository pubblicato.

In [ ]:
# ✏️ IL TUO CODICE QUI
# !git remote add origin git@github.com:TUO-UTENTE/sistema-biblioteca.git
# !git push -u origin main
# !git push --tags


---
### Esercizio 5 (sfida) — Generatore di CHANGELOG

**Consegna:** Scrivi la funzione `generar_changelog(ruta)` che legga la cronologia Git e generi automaticamente un file `CHANGELOG.md` con formato professionale:

```markdown
# Changelog

## [v1.2.0] — 2024-01-15

### Nuove funzionalità
- feat: statistiche avanzate dei prestiti

### Correzioni
- (nessuna)

## [v1.1.0] — 2024-01-10

### Nuove funzionalità
- feat: ...
...
```

La funzione deve:
- Ottenere i tag ordinati cronologicamente.
- Per ogni coppia di tag consecutivi, elencare i commit tra loro.
- Classificarli in `feat`, `fix`, `docs`, `test`, `otros`.
- Scrivere `CHANGELOG.md` e fare commit con `docs: generar CHANGELOG.md`.

In [ ]:
# ✏️ IL TUO CODICE QUI
import subprocess
from pathlib import Path


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import subprocess, re, os
from pathlib import Path

def generar_changelog(ruta):
    ruta = Path(ruta).expanduser().resolve()
    os.chdir(ruta)

    def git(args):
        r = subprocess.run(["git"] + args, cwd=str(ruta),
                           capture_output=True, text=True)
        return r.stdout.strip()

    # Tag ordinati cronologicamente
    tags_raw = git(["tag", "-l", "--sort=version:refname"]).splitlines()
    if not tags_raw:
        print("Nessun tag — impossibile generare il changelog")
        return

    def commits_entre(desde, hasta):
        rango = f"{desde}..{hasta}" if desde else hasta
        raw = git(["log", rango, "--pretty=format:%s", "--no-merges"])
        return [l for l in raw.splitlines() if l.strip()]

    def clasificar(commits):
        grupos = {"feat": [], "fix": [], "docs": [], "test": [], "otros": []}
        for c in commits:
            match = re.match(r'^(feat|fix|docs|test|chore|refactor|style)[:(]', c)
            clave = match.group(1) if match else "otros"
            if clave not in grupos:
                clave = "otros"
            grupos[clave].append(c)
        return grupos

    def fecha_tag(tag):
        return git(["log", "-1", "--format=%ad", "--date=short", tag])

    secciones = {
        "feat": "Nuove funzionalità",
        "fix": "Correzioni",
        "docs": "Documentazione",
        "test": "Test",
        "otros": "Altre modifiche",
    }

    lineas = ["# Changelog\n", "",
              "Generato automaticamente dalla cronologia Git.\n"]

    pares = [(tags_raw[i-1] if i > 0 else None, tags_raw[i])
             for i in range(len(tags_raw)-1, -1, -1)]  # dal più recente al più vecchio

    for desde, hasta in pares:
        commits = commits_entre(desde, hasta)
        fecha = fecha_tag(hasta)
        lineas.append(f"## [{hasta}] — {fecha}\n")

        grupos = clasificar(commits)
        algo = False
        for clave, titulo in secciones.items():
            if grupos[clave]:
                lineas.append(f"### {titulo}")
                for c in grupos[clave]:
                    lineas.append(f"- {c}")
                lineas.append("")
                algo = True
        if not algo:
            lineas.append("_(nessuna modifica registrata)_\n")

    changelog = "\n".join(lineas)
    Path("CHANGELOG.md").write_text(changelog, encoding="utf-8")
    print(changelog)

    subprocess.run(["git", "add", "CHANGELOG.md"], cwd=str(ruta))
    subprocess.run(["git", "commit", "-m", "docs: generare CHANGELOG.md automaticamente"],
                   cwd=str(ruta))
    print("\n✅ CHANGELOG.md generato e committato")


generar_changelog("~/sistema_biblioteca")
```
</details>

---
## Riflessione finale — Cosa abbiamo costruito?

Lungo questo progetto hai applicato, senza che ti venisse indicato passo dopo passo:

| Abilità | Dove l'hai usata |
|-----------|----------------|
| `git init` + `.gitignore` | Fase 1 |
| Commit convenzionali | Tutte le fasi |
| Feature branch + merge --no-ff | Fasi 2, 3 e 4 |
| `git tag -a` | v1.0.0, v1.1.0, v1.0.1, v1.2.0 |
| Hotfix da un tag | Esercizio 2 |
| `git push` + `git push --tags` | Esercizio 4 |
| Audit con `subprocess` | Esercizio 1 e 5 |
| Generazione del CHANGELOG | Esercizio 5 |

**La cosa più importante non è che il codice funzioni — è che la cronologia racconti una storia.** Chi non ha visto il progetto può leggere `git log --oneline --graph` e capire cosa è stato costruito, in che ordine e perché.

> Il codice è per le macchine. La cronologia Git è per le persone.

---
## Modulo 2 completato — Riepilogo di quanto appreso

```
LEZIONE 1  Git Introduzione
         ├── Problema che Git risolve
         ├── Tre zone: working / staging / repo
         ├── init, add, commit, log
         └── .gitignore

LEZIONE 2  Git Locale Avanzato
         ├── diff, restore, revert, reset
         ├── stash, amend
         ├── log avanzato, blame
         └── tag

LEZIONE 3  GitHub e Remoti
         ├── SSH vs HTTPS
         ├── remote add, push, clone
         ├── fetch vs pull
         └── Fork e upstream

LEZIONE 4  Branch e Collaborazione
         ├── branch, switch
         ├── merge: fast-forward e 3-way
         ├── Risoluzione dei conflitti
         ├── Git flow semplificato
         └── Pull Request

LEZIONE 5  Progetto: Sistema Biblioteca
         ├── Cronologia reale con feature branch
         ├── Hotfix da un tag
         ├── Pubblicazione su GitHub con tag
         └── Audit e CHANGELOG automatici
```

**Prossimo modulo:** Python per i dati — NumPy, Pandas e visualizzazione.